<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
original_row_count = len(cat)
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
# Selecting product names containing "pop", regardless of capitalization
pop_rows = cat[
    cat['product_name'].str.contains('pop', case=False, na=False)
]

# Print the count and show the matching rows
print('Number of Pop-Tart rows:', len(pop_rows))
display(pop_rows)

Number of Pop-Tart rows: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
# Find Pop-Tart rows whose names also have "strawberry".
strawberry_rows = pop_rows[
    pop_rows['product_name'].str.contains('strawberry', case=False, na=False)
]

# Show the strawberry Pop-Tart SKUs and thier names
print('Strawberry Pop-Tart rows:')
display(strawberry_rows[['sku', 'product_name']])

# Find Pop-Tart rows that are not strawberry
other_pop_rows = pop_rows[
    ~pop_rows.index.isin(strawberry_rows.index)
]

# Show other Pop-Tart products (these should stay separate)
print('Other Pop-Tart rows that should stay separate:')
display(other_pop_rows[['sku', 'product_name']])

# Print Answer
if len(other_pop_rows) > 0:
    other_names = other_pop_rows['product_name'].dropna().unique().tolist()
    print('Pop-Tart rows that are a different product:', ', '.join(other_names))
else:
    print('Pop-Tart rows that are a different product: None found.')

Strawberry Pop-Tart rows:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Other Pop-Tart rows that should stay separate:


,sku,product_name


Pop-Tart rows that are a different product: None found.


**Pop-Tart rows that are a different product:** none of them

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
# Used PT-12 as the canonical SKU for the strawberry Pop-Tart
canonical_sku = 'PT-12'

# Collect the other strawberry Pop-Tart SKUs
variant_skus = strawberry_rows.loc[
    strawberry_rows['sku'] != canonical_sku, 'sku'
].drop_duplicates().tolist()

# Map each strawberry variant SKU to the canonical SKU
sku_map = {
    sku: canonical_sku
    for sku in variant_skus
}

print('SKU mapping:', sku_map)

# Replace mapped SKUs (leave every unmapped SKU as itself)
cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])

# Check the mapped Pop-Tart rows
display(cat.loc[cat['sku'].isin(strawberry_rows['sku']),
                ['sku', 'product_name', 'canonical_sku']])

SKU mapping: {'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}


,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,PT-12
1,SKU#459812,Strawberry Pop-Tart,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
# Show the original category counts
print('Category counts before normalization:')
display(cat['category'].value_counts(dropna=False))

# Remove extra spaces and title-case each category
cat['category_normalized'] = cat['category'].str.strip().str.title()

# Show the cleaned category counts
print('Category counts after normalization:')
display(cat['category_normalized'].value_counts(dropna=False))

# Show the distinct labels before and after normalization
print('Distinct categories before:')
print(sorted(cat['category'].dropna().unique()))

print('Distinct categories after:')
print(sorted(cat['category_normalized'].dropna().unique()))

Category counts before normalization:


,count
category,
Emergency,5
Emergency Supplies,3
EMERGENCY,3
ENTERTAINMENT,3
GROCERY,3
Breakfast,2
entertainment,2
household,2
ELECTRONICS,2


Category counts after normalization:


,count
category_normalized,
Emergency,9
Entertainment,5
Grocery,5
Electronics,4
Household,4
Emergency Supplies,3
Breakfast,3
Baby,3
Snacks,2


Distinct categories before:
['ALCOHOL', 'Alcoholic Beverages', 'Baby', 'Baby Products', 'Breakfast', 'Breakfast Foods', 'ELECTRONICS', 'EMERGENCY', 'ENTERTAINMENT', 'Electronics', 'Emergency', 'Emergency Supplies', 'GROCERY', 'Groceries', 'Grocery', 'HOUSEHOLD', 'Household', 'Snacks', 'baby', 'breakfast', 'electronics', 'emergency', 'entertainment', 'grocery', 'health', 'household', 'snacks']
Distinct categories after:
['Alcohol', 'Alcoholic Beverages', 'Baby', 'Baby Products', 'Breakfast', 'Breakfast Foods', 'Electronics', 'Emergency', 'Emergency Supplies', 'Entertainment', 'Groceries', 'Grocery', 'Health', 'Household', 'Snacks']


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
# Normalize names by lowercasing them, removing punctuation, and sorting its words.

def normalize_name(name):
    if pd.isna(name):
        return ''

    name = name.lower().strip()
    name = pd.Series([name]).str.replace(
        r'[^a-z0-9\s]', '', regex=True
    ).iloc[0]

    words = name.split()
    words.sort()
    return ' '.join(words)

# Create a normalized name for each product
cat['normalized_name'] = cat['product_name'].apply(normalize_name)

# Group products with the same normalized name and keep groups that contain more than one distinct SKU
candidate_groups = (
    cat[cat['normalized_name'] != '']
    .groupby('normalized_name')
    .agg(
        skus=('sku', lambda values: list(pd.unique(values))),
        product_names=('product_name', lambda values: list(pd.unique(values))),
        package_sizes=('package_size', lambda values: list(pd.unique(values))),
        sku_count=('sku', 'nunique')
    )
)

candidate_groups = candidate_groups[candidate_groups['sku_count'] > 1]

# Display possible fragmented product families
candidate_groups

,skus,product_names,package_sizes,sku_count
normalized_name,,,,
24pack beer light,"[BEER-24, LTBEER24]",[Light Beer (24-pack)],[bundle],2
4pack aa batteries,"[BAT-AA4, BATT_AA_4PK]",[AA Batteries 4-Pack],"[4pk, 24 pack]",2
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Board Game - Monopoly]","[1pk, 24-pack]",3
bread white,"[BREAD-WH, WHITEBREAD]",[WHITE BREAD],"[24-pack, 24 pack]",2
butter peanut,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]","[bundle, 1-pack]",2
candles emergency,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]","[1 pack, 24pk]",2
canned soup,"[SOUP-CAN, CNSOUP-01]","[Soup - Canned, Canned Soup]",[single],2
cards playing,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]","[4pk, 24-pack]",2
chips pack variety,"[CHIPS-VP, CHIPVARIETY]","[Variety Pack - Chips, Chips Variety Pack]","[single, 4-pack]",2


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# Confirm that mapping left no rows without a canonical SKU
assert cat['canonical_sku'].notna().all()

# Confirm that the mapping did not add or remove rows
assert len(cat) == original_row_count

print('Mapping check passed: no null canonical SKUs.')
print('Row count is unchanged:', len(cat))

Mapping check passed: no null canonical SKUs.
Row count is unchanged: 45


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Two product families beyond Pop-Tarts that appear fragmented are Monopoly board games and AA batteries. The Monopoly candidates are BG-MONO, MONOPOLY-01, and GAME-MONO. The AA battery candidates are BAT-AA4 and BATT_AA_4PK. I would group each family under one canonical SKU after checking the product details.

**b)** Consolidation becomes a business decision when the names look alike but the catalog details suggest the products might differ. For example, the two AA battery products appear to be the same item based on their names, but one comes in a 4 pack and the other says 24 pack. I would ask the product catalog owner or merchandising team to confirm whether these are the same product or different package sizes before merging them.